# Novel Grammar Induction

**Last updated:** 2026-04-09 — PT

**Track:** Learning | **Sub-Ability:** Language learning

Can the model induce grammar rules of a novel language from examples?
Pre/post learning framework: observe valid sentences, then judge grammaticality.

**Difficulty grid:** grammar complexity × num examples × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_yes_no(response: str) -> str:
    """Extract YES or NO from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Check lines from the end for a short YES/NO answer
    for line in reversed(lines):
        clean = line.strip('"\'').rstrip('.!?, ').lower().strip()
        if clean in ('yes', 'no'):
            return clean
    # Fallback: search entire text for last yes/no
    matches = re.findall(r'\b(yes|no)\b', text.lower())
    return matches[-1] if matches else ''

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/novel_grammar_induction_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/novel_grammar_induction_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
# === Prompt templates ===

PRE_P = """Here are valid sentences in a novel language:
{material}

Is this sentence grammatical in the same language?
  "{test_sentence}"

Reply with ONLY YES or NO."""

STUDY_P = """Here are valid sentences in a novel language:
{material}

Your task: Analyze the grammar of this language thoroughly.
1. Identify the word classes (which words are nouns, verbs, determiners, adjectives, etc.).
2. Determine the word order rules.
3. Identify any agreement patterns (e.g., which determiners go with which nouns, verb suffixes).
4. Check for embedding or nesting patterns.
5. Verify your grammar against EVERY example sentence.

Write a complete, precise grammar description."""

POST_P = """You previously studied a novel language and wrote these grammar notes:

--- YOUR GRAMMAR NOTES ---
{notes}
--- END NOTES ---

Here are the original valid sentences for reference:
{material}

Is this sentence grammatical in the same language?
  "{test_sentence}"

Reply with ONLY YES or NO."""

## Evaluation

In [ ]:
all_results = []
@kbench.task(name='novel_grammar_induction',
             description='Pre/post learning grammar induction from novel language examples')
def novel_grammar_induction(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)

    for _, row in dataset.iterrows():
        material = row['material']
        test_sentence = row['test_sentence']
        expected = row['expected']
        complexity = row['complexity']
        evidence = row['evidence']
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        tid = f'{complexity}_{evidence}_{seed}'

        # --- Pre-learning: judge grammaticality without study ---
        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(material=material, test_sentence=test_sentence))
        pre_time = time.time() - t0
        pre_extracted = extract_yes_no(pre_raw)
        pre_correct = pre_extracted == expected

        # --- Study phase: analyze the grammar ---
        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(material=material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        # --- Post-learning: judge with notes ---
        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_sentence=test_sentence))
        post_time = time.time() - t0
        post_extracted = extract_yes_no(post_raw)
        post_correct = post_extracted == expected

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': tid,
            'complexity': complexity, 'evidence': evidence, 'difficulty_label': difficulty_label,
            'seed': int(seed), 'item_desc': item_desc,
            'test_sentence': test_sentence, 'expected': expected,
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
        })

        b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
        print(f'[{difficulty_label:12s}] "{test_sentence}"  exp={expected}  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})  times: {pre_time:.1f}s/{study_time:.1f}s/{post_time:.1f}s')
        # Per-item assertion for diagnostics
        kbench.assertions.assert_equal(post_extracted, expected)

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = novel_grammar_induction.evaluate(llm=[kbench.llm],
                                            n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:12s}] "{row["test_sentence"]}"  exp={row["expected"]}  '
          f'pre="{row["pre_extracted"]}"({b})  post="{row["post_extracted"]}"({l}){delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Test: "{row["test_sentence"]}"  Expected: {row["expected"]}')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Pre vs Post-Learning: Grammar Induction')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('grammar_induction_results.png', dpi=150, bbox_inches='tight')
plt.show()